# RNN Architecture Comparison
**Winner: Vanilla RNN (0.2175) | Worst: BiLSTM (0.2312)**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping
from utils.training import train_two_stage, get_predictions
from utils.reproducibility_utils import seed_everything

# Models
from models.vanilla_rnn import VanillaRNN
from models.simple_lstm import SimpleLSTM
from models.simple_gru import SimpleGRU
from models.simple_bilstm import SimpleBiLSTM
from models.bilstm_attention import BiLSTMAttention

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

## Architecture Comparison

In [ ]:
architectures = [
    ('VanillaRNN', VanillaRNN(22, 64)),
    ('LSTM', SimpleLSTM(22, 64)),
    ('GRU', SimpleGRU(22, 64)),
    ('BiLSTM', SimpleBiLSTM(22, 64)),
    ('BiLSTM+Attention', BiLSTMAttention(22, 64, 2, 4)),
]

results = []
for name, model in architectures:
    seed_everything(SEED)
    params = sum(p.numel() for p in model.parameters())
    print(f"\nTraining {name} ({params:,} params)...")
    
    _, metrics = train_two_stage(model, train_loader, val_loader, band_dict, device)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    results.append({'model': name, 'params': params, **metrics})

pd.DataFrame(results).sort_values('overall_mae')

## Predictions vs Ground Truth

In [ ]:
import plotly.graph_objects as go

# Train winner
seed_everything(SEED)
winner = VanillaRNN(22, 64)
winner, _ = train_two_stage(winner, train_loader, val_loader, band_dict, device)

val_preds, val_targets = get_predictions(winner, val_loader, device)

fig = go.Figure()
fig.add_trace(go.Scatter(y=val_targets, mode='lines', name='Ground Truth', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=val_preds, mode='lines', name='VanillaRNN', line=dict(color='red'), opacity=0.8))
fig.update_layout(title='Vanilla RNN: Ground Truth vs Predictions', xaxis_title='Sample', yaxis_title='Sideslip (°)', width=1200, height=500)
fig.show()

## Summary
| Finding | Result |
|---------|--------|
| Best architecture | Vanilla RNN (64h, 1L) |
| Bidirectional benefit | None (20 timesteps too short) |
| Attention benefit | Marginal |
| More params | Worse (overfitting) |